# 🚀 Otimização de Hiperparâmetros - SVM

Este notebook apresenta um pipeline completo de Machine Learning focado na maximização da acurácia para o dataset Titanic. A estratégia central baseia-se em um **fluxo de refinamento em três etapas**, movendo-se de uma exploração global para uma otimização fina e localizada.

### 🏗️ Estrutura do Pipeline

1. **Pré-processamento Customizado (`PP2`):** Utilização de um pipeline de engenharia de atributos previamente validado e serializado (v1.2), garantindo a consistência dos dados entre treino e teste.
2. **Modelo Base:** Implementação do `SVM` como estimador central, encapsulado em um `Pipeline` do Scikit-Learn.
3. **Otimização de Threshold:** Diferente da predição padrão (0.5), o código implementa uma busca exaustiva pelo **limiar de decisão ideal**, ajustando a sensibilidade do modelo para maximizar a acurácia final.

### 🔍 Estratégia de Busca (O Funil)

O processo de busca de hiperparâmetros foi dividido em três níveis de granularidade:

* **Fase 1: Busca Exploratória (`RandomizedSearchCV`):** Exploração de um espaço amostral amplo para identificar as regiões de alta performance e os parâmetros mais influentes.
* **Fase 2: Refinamento Estruturado (`GridSearchCV`):** Estreitamento dos intervalos de busca com base nos resultados da Fase 1, aumentando a densidade de tentativas em áreas promissoras.

### 📊 Avaliação e Robustez

A decisão final não se baseia apenas em um único score, mas em uma bateria de testes rigorosos:

* **Métricas Cruzadas:** Comparação de `ROC-AUC` e `Accuracy` via Cross-Validation (10-folds).
* **Estabilidade Estatística:** Aplicação do **T-Test Pareado** para confirmar se a diferença de performance entre os modelos é estatisticamente significante ou fruto do acaso.
* **Generalização:** Teste final em dados inéditos (`X_test`) comparando a acurácia padrão (0.5) contra a acurácia com threshold otimizado.

### 💾 Entrega

O código final automatiza o salvamento dos  estimadores(modelos) em formato `.joblib`, permitindo a reprodutibilidade imediata e o deploy dos modelos otimizados.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

# SVM
from sklearn.svm import SVC

# sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_score, RandomizedSearchCV,cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin, clone
from scipy.stats import loguniform, randint, ttest_rel, uniform



#hiperparamentros search
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV
from scipy.stats import randint
from scipy.stats import randint, uniform


# Importações locais 
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_tic import preprocessador_titanic
from src.plot_metrica_class import *

# Configurações e Inicialização
warnings.filterwarnings("ignore")
print(f"\n# Processo iniciado em: {time.strftime('%H:%M:%S')}")


# Processo iniciado em: 11:40:37


## 1. Load Data & Pipeline

In [2]:
BASE = Path.cwd().parent

PP2 = joblib.load(BASE/'src'/'preprocess_Titanic_v1.2.joblib')['preprocessador']

DATA_DIR = BASE/"data"/"raw"
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_test  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_test  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()
mtd_scoring='accuracy'
print(f"\n# Processo iniciado em: {time.strftime('%H:%M:%S')}")


# Processo iniciado em: 11:40:37


## 2. Baseline

In [3]:
# Baseline
model_base = SVC(kernel='rbf',random_state=42, probability=True)  # kernel não fixado!

pipe_base = Pipeline([('preprocess', PP2),
                      ('scaler', StandardScaler()),
                      ('model', model_base)])

pipe_base.fit(X_train, y_train)

# ==============================================================================
# CORREÇÃO: best_threshold2-THRESHOLD NO TREINO E PROBS NO TESTE(20/02/2026)
# ==============================================================================
best_th0,max_acc0,y_probs0=best_threshold2(pipe_base, X_train, y_train,X_test,y_test)
# ==============================================================================


print(f"{'='*70}")
print(f"🎯 Melhor Threshold: {best_th0:.3f}")
print(f"📈 Melhor Acurácia de Teste: {max_acc0:.4f}")

baseline_scores = cross_val_score(pipe_base, X_train, y_train, cv=10)
print(f"Baseline: {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f}")
print(f"Average CV Accuracy: {np.mean(baseline_scores)*100:.2f}%") 

mtd_scoring='accuracy'
#accuracy

# linear=================================================================
#🎯 Melhor Threshold: 0.560
#📈 Melhor Acurácia de Teste: 0.8284
#Baseline: 0.8154 ± 0.0591
#Average CV Accuracy: 81.54%
# poly===================================================================
# 🎯 Melhor Threshold: 0.480
# 📈 Melhor Acurácia de Teste: 0.8060
# Baseline: 0.8171 ± 0.0418
# Average CV Accuracy: 81.71%
# rbf===================================================================
# 🎯 Melhor Threshold: 0.370
# 📈 Melhor Acurácia de Teste: 0.8209
# Baseline: 0.8379 ± 0.0629
# Average CV Accuracy: 83.79%

🎯 Melhor Threshold: 0.550
📈 Melhor Acurácia de Teste: 0.8134
Baseline: 0.8379 ± 0.0629
Average CV Accuracy: 83.79%


## 3.Buscas por hiperparamentros
### 3.1.Random Search (Exploratória)

In [4]:
print(f"# Processo iniciado em: {time.strftime('%H:%M:%S')}")
param_dist_1 = {
    'model__gamma':  uniform(0.01, 0.08), 
    'model__C': loguniform(0.1,8),              
    'model__class_weight': [None]
}


n_it=100
search_1 =  RandomizedSearchCV(
    pipe_base,
    param_dist_1,
    n_iter=n_it, 
    cv=10,
    scoring=mtd_scoring,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

start = time.time()
search_1.fit(X_train, y_train)
end = time.time()

best_1 = search_1.best_estimator_

# 2. Testar vários thresholds
best_th1,max_acc1,y_probs1 =best_threshold2(best_1 , X_train, y_train,X_test,y_test)


print("📌 Melhores Parâmetros:")
print(search_1.best_params_)
print(f"\n{'='*70}")
print(f"🎯 Melhor Threshold: {best_th1:.3f}")
print(f"📈 Melhor Acurácia de Teste: {max_acc1:.4f}")
print(f"{'='*70}")

#ACCURACY
scores1 = cross_val_score(best_1, X_train, y_train, cv=10)
print(f"{'='*70}")
print(f"Optimized: {scores1.mean():.4f} ± {scores1.std():.4f}")
print(f"Average CV Accuracy: {np.mean(scores1)*100:.2f}%")
print(f"Tempo total: {end-start:.2f} segundos")
print(f"Tempo por iteração: {(end-start)/n_it:.2f} segundos")
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

# Processo iniciado em: 11:40:38
📌 Melhores Parâmetros:
{'model__C': 2.94884053630599, 'model__class_weight': None, 'model__gamma': 0.025897254522733795}

🎯 Melhor Threshold: 0.680
📈 Melhor Acurácia de Teste: 0.8134
Optimized: 0.8476 ± 0.0483
Average CV Accuracy: 84.76%
Tempo total: 35.10 segundos
Tempo por iteração: 0.35 segundos

#Processo finalizado em: 11:41:14


### 3.2 SVM (Refino)

In [8]:
from sklearn.model_selection import GridSearchCV

C_s1=search_1.best_params_['model__C']
gamma_s1=search_1.best_params_['model__gamma']

C_values = np.logspace(np.log10(C_s1-1), np.log10(C_s1+1), num=20)
gamma_values = np.linspace(gamma_s1-0.01, gamma_s1+0.01, num=15)

param_grid = {
    'model__gamma': gamma_values,
    'model__C': C_values,
    'model__class_weight': [None]
}

print(f"# Processo iniciado em: {time.strftime('%H:%M:%S')}")
print(f"\n📊 Grid de parâmetros:")
print(f"   C: {len(C_values)} valores (de {C_values[0]:.3f} a {C_values[-1]:.3f})")
print(f"   Gamma: {len(gamma_values)} valores (de {gamma_values[0]:.3f} a {gamma_values[-1]:.3f})")
print(f"   Total combinações: {len(C_values) * len(gamma_values)}")


grid_search = GridSearchCV(
    pipe_base,
    param_grid,
    cv=10,
    scoring=mtd_scoring,
    n_jobs=-1,
    verbose=1
)
start = time.time()
grid_search.fit(X_train, y_train)
end = time.time()

best_2 = grid_search.best_estimator_

# 2. Testar vários thresholds
best_th2,max_acc2,y_probs2 =best_threshold2(best_2 , X_train, y_train,X_test,y_test)

print("📌 Melhores Parâmetros:")
print(grid_search.best_params_)
print(f"\n{'='*70}")
print(f"🎯 Melhor Threshold: {best_th2:.3f}")
print(f"📈 Melhor Acurácia de Teste: {max_acc2:.4f}")
print(f"{'='*70}")

#ACCURACY
scores2 = cross_val_score(best_2, X_train, y_train, cv=10)
print(f"{'='*70}")
print(f"Optimized: {scores2.mean():.4f} ± {scores2.std():.4f}")
print(f"Average CV Accuracy: {np.mean(scores2)*100:.2f}%")
print(f"Tempo total: {end-start:.2f} segundos")
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))


# Processo iniciado em: 11:43:58

📊 Grid de parâmetros:
   C: 20 valores (de 1.949 a 3.949)
   Gamma: 15 valores (de 0.016 a 0.036)
   Total combinações: 300
Fitting 10 folds for each of 300 candidates, totalling 3000 fits
📌 Melhores Parâmetros:
{'model__C': 1.9488405363059902, 'model__class_weight': None, 'model__gamma': 0.03446868309416237}

🎯 Melhor Threshold: 0.690
📈 Melhor Acurácia de Teste: 0.8097
Optimized: 0.8476 ± 0.0483
Average CV Accuracy: 84.76%
Tempo total: 102.32 segundos

#Processo finalizado em: 11:45:41


## 4. Avaliando resultados

In [9]:
# Calcula os scores de validação cruzada para cada modelo(roc_auc)

s0 = cross_val_score(pipe_base, X_train, y_train, cv=10,scoring='roc_auc')
s1 = cross_val_score(best_1, X_train, y_train, cv=10,scoring='roc_auc')
s2 = cross_val_score(best_2, X_train, y_train, cv=10,scoring='roc_auc')

# Calcula os scores de validação cruzada para cada modelo(acc)
s0_acc = cross_val_score(pipe_base, X_train, y_train, cv=10)
s1_acc = cross_val_score(best_1, X_train, y_train, cv=10)
s2_acc= cross_val_score(best_2, X_train, y_train, cv=10)

best_1.fit(X_train, y_train)
best_2.fit(X_train, y_train)

score1 = best_1.score(X_test, y_test)
score2 = best_2.score(X_test, y_test)


# 1. Preparação dos Dados de Performance
models_list = [('Modelo 0 (baseln)', pipe_base, s0, s0_acc, y_probs0, best_th0),
               ('Modelo 1 (Random)', best_1, s1, s1_acc, y_probs1, best_th1),
               ('Modelo 2 (Refine)', best_2, s2, s2_acc, y_probs2, best_th2),
]

df_results, W = gerar_relatorio_estatistico(models_list,
                                            X_train, y_train,
                                            X_test, y_test)

                      RELATÓRIO DE DESEMPENHO E ESTABILIDADE ESTATÍSTICA                       
      Modelo         CV ROC-AUC        CV ACC       Test ROC-AUC   Test ACC (0.5)   Best Thresh    Test ACC (Opt)
Modelo 0 (baseln) 0.8513 ± 0.0596 0.8379 ± 0.0629      0.8769          0.8209          0.550           0.8134    
Modelo 1 (Random) 0.8531 ± 0.0593 0.8476 ± 0.0483      0.8649          0.8209          0.680           0.8134    
Modelo 2 (Refine) 0.8488 ± 0.0634 0.8476 ± 0.0483      0.8734          0.8172          0.690           0.8097    

                     ANÁLISE DE SIGNIFICÂNCIA ESTATÍSTICA (T-TEST PAREADO)                     
Modelo 1 (Random) vs Modelo 2 (Refine): p-value = 0.0691 | Diferença Significativa? NÃO

                                 CONCLUSÃO TÉCNICA AUTOMÁTICA                                  
1. VENCEDOR: Modelo 0 (baseln)
   - Ganho real sobre o Baseline: +0.0000 em Test ROC-AUC.

2. ESTABILIDADE E SIGNIFICÂNCIA:
   - A melhoria em relação ao Baseline é n

### 5. Salvar modelo

In [10]:
# Salvar Hiperparametros joblib
# apenas parametros
#joblib.dump(bayes_search.best_params_, 'parametros_RF_BAYER_v12.joblib') 

# modelo completo
joblib.dump(search_1.best_estimator_, 'modelo_SVM_final_randsearch.'+mtd_scoring+'_v12.joblib')
joblib.dump(grid_search.best_estimator_, 'modelo_SVM_final_gridsearch.'+mtd_scoring+'_v12.joblib')
print("\n#Arquivos salvos", time.strftime("%H:%M:%S"))


#Arquivos salvos 11:53:06
